# Lessons 12-13: RAG answers + LLM as a judge

Free tier: we run RAG + judge on a **15-question sample** with Groq. Full-size numbers come from the course files (395 answers, judged with gpt-5.4-mini). Extra: we judge 15 of the course answers with our Groq judge and compare verdicts with the course judge.

In [1]:
import os, json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv("../.env")

# Course: openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1",
)

In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("data/course/ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []
for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)
len(documents)

161

In [4]:
doc_idx = {}
for doc in documents:
    doc_idx[doc["id"]] = doc

# keep only questions whose document is still in today's FAQ
ground_truth = [r for r in ground_truth if r["document"] in doc_idx]
len(ground_truth)

315

## RAG answers (lesson 12)

In [5]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage(index=index, llm_client=openai_client)

rec = ground_truth[0]
answer_llm = assistant.rag(rec["question"])
print(rec["question"])
print("---")
print(answer_llm)
print("---")
print(doc_idx[rec["document"]]["answer"])
assistant.total_cost()

Is it okay to join the course late if I just found it now?
---
Yes—you can still join the course even if you just discovered it. The only catch is that if you want to earn a certificate you’ll need to submit your capstone project before the submission window closes. Otherwise you can start learning and doing the homework right away.
---
Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.


0.0001467

In [6]:
def generate_rag_answer(rec):
    question = rec["question"]
    doc_id = rec["document"]
    answer_llm = assistant.rag(question)
    return {"question": question, "answer_llm": answer_llm,
            "answer_orig": doc_idx[doc_id]["answer"], "document": doc_id}

In [7]:
import random
from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress
import time

random.seed(42)
sample = random.sample(ground_truth, 15)

assistant.reset_usage()
t0 = time.time()
with ThreadPoolExecutor(max_workers=2) as pool:
    answers = map_progress(pool, sample, generate_rag_answer)
print(f"{time.time() - t0:.0f} s, cost $", round(assistant.total_cost(), 6))

df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-groq-sample.csv", index=False)

  0%|          | 0/15 [00:00<?, ?it/s]

121 s, cost $ 0.005078


## LLM as a judge (lesson 13)

In [8]:
from pydantic import BaseModel, Field
from typing import Literal

class AnswerEvaluation(BaseModel):
    reasoning: str = Field(description="Reasoning about the quality of the answer.")
    score: Literal["good", "bad"] = Field(description="'good' if the answer is correct and complete, 'bad' otherwise.")

In [9]:
aqa_judge_instructions = """
You are an expert evaluator. You will be given:
1. A question from a student
2. The original answer from the FAQ (ground truth)
3. An answer generated by an AI assistant

Your task is to decide if the AI answer is semantically equivalent to
the original answer.

Rules:
- The AI answer does NOT need to be word-for-word identical
- It should convey the same key information
- Extra detail is fine as long as the core answer is correct
- Mark 'bad' only if the AI answer is wrong or misses the key point

Be fair and focus on correctness, not style.
""".strip()

aqa_judge_prompt = """
Question:
{question}

Original Answer (ground truth):
{answer_orig}

AI Answer:
{answer_llm}
""".strip()

In [10]:
from evaluation_utils import calc_price, calc_total_price, llm_structured_retry

def evaluate_aqa(question, answer_orig, answer_llm):
    prompt = aqa_judge_prompt.format(question=question, answer_orig=answer_orig, answer_llm=answer_llm)
    return llm_structured_retry(openai_client, aqa_judge_instructions, prompt, AnswerEvaluation)

rec = answers[0]
eval_result, usage = evaluate_aqa(rec["question"], rec["answer_orig"], rec["answer_llm"])
print(eval_result.score, "|", eval_result.reasoning)
calc_price(usage)

good | The AI answer correctly identifies that the error message indicates insufficient credits, which matches the original answer. It expands on the original by suggesting specific checks (account balance, adding funds, model access, API key) but still conveys the core information that the error is due to lack of credits. Therefore it is semantically equivalent and correct.


{'input_cost': 0.00011865, 'output_cost': 0.0001518, 'total_cost': 0.00027045}

In [11]:
def judge_record(rec):
    eval_result, usage = evaluate_aqa(rec["question"], rec["answer_orig"], rec["answer_llm"])
    return {"question": rec["question"], "document": rec["document"],
            "score": eval_result.score, "reasoning": eval_result.reasoning}, usage

with ThreadPoolExecutor(max_workers=2) as pool:
    results = map_progress(pool, answers, judge_record)

evaluations = [e for e, u in results]
usages = [u for e, u in results]
df_eval = pd.DataFrame(evaluations)
good = (df_eval["score"] == "good").sum()
print(f"Groq RAG + Groq judge: good {good}/{len(df_eval)} = {good/len(df_eval):.0%}, judge cost $", round(calc_total_price(usages), 6))
df_eval.to_csv("data/rag-evaluations-groq-sample.csv", index=False)

  0%|          | 0/15 [00:00<?, ?it/s]

Groq RAG + Groq judge: good 15/15 = 100%, judge cost $ 0.003723


In [12]:
df_eval[df_eval["score"] == "bad"][["question", "reasoning"]]

,question,reasoning


## Course full results (gpt-5.4-mini RAG + judge, 395 answers)

In [13]:
df_course_eval = pd.read_csv("data/course/rag-evaluations-new.csv")
df_course_eval["score"].value_counts()

score
good    379
bad      16
Name: count, dtype: int64

## Evaluating the judge: our Groq judge vs the course judge on the same 15 course answers

In [14]:
df_course_answers = pd.read_csv("data/course/rag-answers-new.csv")
course_sample = df_course_answers.sample(15, random_state=1).to_dict(orient="records")

with ThreadPoolExecutor(max_workers=2) as pool:
    results = map_progress(pool, course_sample, judge_record)

ours = pd.DataFrame([e for e, u in results])[["question", "score"]].rename(columns={"score": "groq_judge"})
theirs = df_course_eval[["question", "score"]].drop_duplicates("question").rename(columns={"score": "course_judge"})
cmp = ours.merge(theirs, on="question", how="left")
agree = (cmp["groq_judge"] == cmp["course_judge"]).sum()
print(f"agreement {agree}/{len(cmp)}")
cmp[cmp["groq_judge"] != cmp["course_judge"]]

  0%|          | 0/15 [00:00<?, ?it/s]

agreement 14/15


,question,groq_judge,course_judge
5,Which free Groq models support tool use right ...,bad,good
